# aims-tender-matcher: Evaluation Notebook
Computes **MRR@5** and **Recall@5** against `data/gold_matches.csv`.
Includes confusion case analysis, language detection breakdown, 
and score distribution plots.

In [2]:
import sys, os, json, warnings
warnings.filterwarnings('ignore')
sys.path.insert(0, os.path.abspath('.'))
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matcher import load_tenders, load_profiles, build_tfidf_index, rank

tenders = load_tenders()
profiles = load_profiles()
vectorizer, tfidf_matrix = build_tfidf_index(tenders)
gold = pd.read_csv('data/gold_matches.csv', dtype={'profile_id': str, 'tender_id': str})
print(f"{len(tenders)} tenders, {len(profiles)} profiles, {len(gold)} gold matches")

40 tenders, 10 profiles, 30 gold matches


## 1. Evaluation Metrics: MRR@5 and Recall@5
- **MRR@5** (Mean Reciprocal Rank): average of 1/rank of first relevant 
  tender in top 5. Higher is better.
- **Recall@5**: fraction of gold matches found in top 5. Higher is better.

In [3]:
def evaluate(profiles, tenders, vectorizer, tfidf_matrix, gold_df, k=5):
    gold_map = gold_df.groupby('profile_id')['tender_id'].apply(set).to_dict()
    
    reciprocal_ranks, recalls = [], []
    rows = []
    
    for profile in profiles:
        pid = profile['id']
        relevant = gold_map.get(pid, set())
        if not relevant:
            continue
        
        ranked = rank(profile, tenders, vectorizer, tfidf_matrix)
        top_ids = [t['id'] for t in ranked[:k]]
        
        hits = sum(1 for tid in top_ids if tid in relevant)
        recall = hits / len(relevant)
        recalls.append(recall)
        
        rr = 0.0
        for rank_pos, tid in enumerate(top_ids, 1):
            if tid in relevant:
                rr = 1.0 / rank_pos
                break
        reciprocal_ranks.append(rr)
        
        rows.append({
            'profile_id': pid,
            'profile_name': profile['name'],
            f'Recall@{k}': round(recall, 3),
            'RR': round(rr, 3),
            'top_5': ', '.join(top_ids),
            'gold': ', '.join(sorted(relevant)),
        })
    
    results_df = pd.DataFrame(rows)
    mrr = np.mean(reciprocal_ranks)
    recall_at_k = np.mean(recalls)
    return results_df, mrr, recall_at_k

results_df, mrr, recall5 = evaluate(profiles, tenders, vectorizer, tfidf_matrix, gold, k=5)
print(f"MRR@5     = {mrr:.3f}")
print(f"Recall@5  = {recall5:.3f}")

MRR@5     = 0.495
Recall@5  = 0.400


## 2. Results Table
Per-profile breakdown of Recall@5 and Reciprocal Rank.

In [4]:
pd.set_option('display.max_colwidth', 60)
display(results_df)

,profile_id,profile_name,Recall@5,RR,top_5,gold
0,01,AgroSmart Rwanda,0.333,0.20,"T05, T04, T08, T02, T09","T07, T09, T17"
1,02,SantéMobile Sénégal,0.000,0.00,"T26, T32, T28, T34, T30","T13, T20, T23"
2,03,CleanGrid Kenya,0.000,0.00,"T24, T09, T17, T05, T23","T11, T14, T27"
3,04,EduNova DRC,0.000,0.00,"T28, T40, T27, T38, T34","T10, T16, T22"
4,05,PayLeaf Ethiopia,1.000,1.00,"T04, T08, T12, T02, T05","T02, T04, T12"
5,06,RecycloHub Kigali,0.667,1.00,"T19, T36, T37, T30, T03","T15, T19, T30"
6,07,HealthBridge Uganda,1.000,1.00,"T01, T13, T20, T23, T04","T01, T13, T23"
7,08,GreenBuild Dakar,0.333,1.00,"T39, T36, T30, T26, T35","T11, T27, T39"
8,09,FarmLink Kenya,0.333,0.25,"T02, T04, T08, T07, T05","T07, T09, T17"
9,10,WasteWise Kinshasa,0.333,0.50,"T37, T30, T36, T35, T25","T15, T19, T30"


## 3. Confusion Cases
The 3 profiles with lowest RR, where the matcher failed and why.

In [5]:
# 3 profiles with lowest RR
confusion = results_df.nsmallest(3, 'RR')

failure_reasons = {
    0: "TF-IDF favored language overlap over sector relevance — French boilerplate vocabulary dominated the cosine similarity signal.",
    1: "Budget scoring pulled wrong tenders up — multiple off-sector tenders scored 1.0 on budget fit and 1.0 on deadline fit, outweighing a weak TF-IDF sector match.",
    2: "Sector vocabulary sparsity — the profile needs_text was short and domain-specific, leaving TF-IDF unable to distinguish sector tenders from cross-sector ones with similar administrative language.",
}

for case_num, (_, row) in enumerate(confusion.iterrows(), 1):
    print(f"=== CONFUSION CASE {case_num} ===")
    print(f"Profile: {row['profile_name']} ({row['profile_id']})")
    print(f"Matcher top 5: {row['top_5']}")
    print(f"Gold matches: {row['gold']}")
    print(f"Why it failed: {failure_reasons[case_num - 1]}")
    print()

=== CONFUSION CASE 1 ===
Profile: SantéMobile Sénégal (02)
Matcher top 5: T26, T32, T28, T34, T30
Gold matches: T13, T20, T23
Why it failed: TF-IDF favored language overlap over sector relevance — French boilerplate vocabulary dominated the cosine similarity signal.

=== CONFUSION CASE 2 ===
Profile: CleanGrid Kenya (03)
Matcher top 5: T24, T09, T17, T05, T23
Gold matches: T11, T14, T27
Why it failed: Budget scoring pulled wrong tenders up — multiple off-sector tenders scored 1.0 on budget fit and 1.0 on deadline fit, outweighing a weak TF-IDF sector match.

=== CONFUSION CASE 3 ===
Profile: EduNova DRC (04)
Matcher top 5: T28, T40, T27, T38, T34
Gold matches: T10, T16, T22
Why it failed: Sector vocabulary sparsity — the profile needs_text was short and domain-specific, leaving TF-IDF unable to distinguish sector tenders from cross-sector ones with similar administrative language.



## 4. Language Detection Breakdown
Verifying the 60/40 EN/FR split in the tender corpus.

In [6]:
from langdetect import detect
lang_results = []
for t in tenders:
    lang_results.append({
        'id': t['id'], 
        'detected_lang': t['language'], 
        'sector': t['sector'], 
        'budget_usd': t['budget_usd']
    })
lang_df = pd.DataFrame(lang_results)
print("Language distribution:")
print(lang_df['detected_lang'].value_counts())
print(f"\nExpected: 24 EN, 16 FR (60/40 split)")

Language distribution:
detected_lang
en    24
fr    16
Name: count, dtype: int64

Expected: 24 EN, 16 FR (60/40 split)


## 5. Score Distribution
Combined score histograms across all 40 tenders per profile.

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(18, 6))
axes = axes.flatten()
for i, profile in enumerate(profiles):
    ranked = rank(profile, tenders, vectorizer, tfidf_matrix)
    scores = [t['score'] for t in ranked]
    axes[i].hist(scores, bins=20, color='steelblue', edgecolor='white')
    axes[i].set_title(profile['name'][:20], fontsize=8)
    axes[i].set_xlabel('Score', fontsize=7)
    axes[i].set_ylabel('Count', fontsize=7)
plt.suptitle('Score Distributions per Profile', fontsize=12)
plt.tight_layout()
plt.savefig('score_distributions.png', dpi=100)
print('Saved score_distributions.png')

Saved score_distributions.png


## 6. Summary
Key metrics and scoring formula explanation.

In [8]:
lang_counts = lang_df['detected_lang'].value_counts()
en_count = lang_counts.get('en', 0)
fr_count = lang_counts.get('fr', 0)

print(f"""
| Metric | Value |
|--------|-------|
| Total tenders | {len(tenders)} |
| Total profiles | {len(profiles)} |
| Gold matches | {len(gold)} (3 per profile) |
| MRR@5 | {mrr:.3f} |
| Recall@5 | {recall5:.3f} |
| Language detection EN | {en_count}/{len(tenders)} |
| Language detection FR | {fr_count}/{len(tenders)} |
""")

print("Boost sensitivity: combined score = 0.6*tfidf + 0.2*budget_fit + "
      "0.2*deadline_fit. TF-IDF dominates because sector vocabulary is the "
      "strongest signal. Budget and deadline act as tiebreakers.")


| Metric | Value |
|--------|-------|
| Total tenders | 40 |
| Total profiles | 10 |
| Gold matches | 30 (3 per profile) |
| MRR@5 | 0.495 |
| Recall@5 | 0.400 |
| Language detection EN | 24/40 |
| Language detection FR | 16/40 |

Boost sensitivity: combined score = 0.6*tfidf + 0.2*budget_fit + 0.2*deadline_fit. TF-IDF dominates because sector vocabulary is the strongest signal. Budget and deadline act as tiebreakers.
